<a href="https://colab.research.google.com/github/llazdll/spark/blob/main/BusyMonth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
pip install pyspark

In [35]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, format_number, concat, substring, to_timestamp, date_format,count


In [9]:
spark=SparkSession.builder.appName("BusyMonth").master("local[*]").getOrCreate()

In [10]:
spark

In [6]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/content/drive/MyDrive/lax_passengers_header.csv")

In [11]:
df.show(20)

+--------------------+--------------------+-----------------+-----------------+----------------------+---------------+
|     DataExtractDate|        ReportPeriod|         Terminal|Arrival_Departure|Domestic_International|Passenger_Count|
+--------------------+--------------------+-----------------+-----------------+----------------------+---------------+
|05/01/2014 12:00:...|01/01/2006 12:00:...|Imperial Terminal|          Arrival|              Domestic|            490|
|05/01/2014 12:00:...|01/01/2006 12:00:...|Imperial Terminal|        Departure|              Domestic|            498|
|05/01/2014 12:00:...|01/01/2006 12:00:...|   Misc. Terminal|          Arrival|              Domestic|            753|
|05/01/2014 12:00:...|01/01/2006 12:00:...|   Misc. Terminal|        Departure|              Domestic|            688|
|05/01/2014 12:00:...|01/01/2006 12:00:...|       Terminal 1|          Arrival|              Domestic|         401535|
|05/01/2014 12:00:...|01/01/2006 12:00:...|     

In [12]:
month_col = "ReportPeriod"
terminal_col = "Terminal"
passenger_col = "Passenger_Count"
terminals = [f"Terminal {i}" for i in range(1, 9)] + ["Tom Bradley International Terminal"]


In [13]:
filtered_df = df.filter(col(terminal_col).isin(terminals))


In [14]:
filtered_df.select(
    "ReportPeriod",
    "Terminal",
    "Passenger_Count"
).show()

+--------------------+----------+---------------+
|        ReportPeriod|  Terminal|Passenger_Count|
+--------------------+----------+---------------+
|01/01/2006 12:00:...|Terminal 1|         401535|
|01/01/2006 12:00:...|Terminal 1|         389745|
|01/01/2006 12:00:...|Terminal 1|            561|
|01/01/2006 12:00:...|Terminal 2|          98991|
|01/01/2006 12:00:...|Terminal 2|         163067|
|01/01/2006 12:00:...|Terminal 2|          93672|
|01/01/2006 12:00:...|Terminal 2|         156751|
|01/01/2006 12:00:...|Terminal 3|         121649|
|01/01/2006 12:00:...|Terminal 3|          26585|
|01/01/2006 12:00:...|Terminal 3|         120111|
|01/01/2006 12:00:...|Terminal 3|          60948|
|01/01/2006 12:00:...|Terminal 4|         381419|
|01/01/2006 12:00:...|Terminal 4|          68348|
|01/01/2006 12:00:...|Terminal 4|         374238|
|01/01/2006 12:00:...|Terminal 4|          42256|
|01/01/2006 12:00:...|Terminal 5|         135622|
|01/01/2006 12:00:...|Terminal 5|          32634|


In [19]:
df2 = filtered_df.withColumn(
    "ReportPeriod_ts",
    to_timestamp(
        "ReportPeriod",
        "MM/dd/yyyy hh:mm:ss a"
    )
)
df2.show()

+--------------------+--------------------+----------+-----------------+----------------------+---------------+-------------------+
|     DataExtractDate|        ReportPeriod|  Terminal|Arrival_Departure|Domestic_International|Passenger_Count|    ReportPeriod_ts|
+--------------------+--------------------+----------+-----------------+----------------------+---------------+-------------------+
|05/01/2014 12:00:...|01/01/2006 12:00:...|Terminal 1|          Arrival|              Domestic|         401535|2006-01-01 00:00:00|
|05/01/2014 12:00:...|01/01/2006 12:00:...|Terminal 1|        Departure|              Domestic|         389745|2006-01-01 00:00:00|
|05/01/2014 12:00:...|01/01/2006 12:00:...|Terminal 1|        Departure|         International|            561|2006-01-01 00:00:00|
|05/01/2014 12:00:...|01/01/2006 12:00:...|Terminal 2|          Arrival|              Domestic|          98991|2006-01-01 00:00:00|
|05/01/2014 12:00:...|01/01/2006 12:00:...|Terminal 2|          Arrival|    

In [20]:
df3 = df2.withColumn(
    "Month",
    date_format("ReportPeriod_ts", "MM/yyyy")
)


In [21]:
df3.select(
    "ReportPeriod",
    "Month",
    "Terminal",
    "Passenger_Count"
).show(20, truncate=False)

+----------------------+-------+----------+---------------+
|ReportPeriod          |Month  |Terminal  |Passenger_Count|
+----------------------+-------+----------+---------------+
|01/01/2006 12:00:00 AM|01/2006|Terminal 1|401535         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 1|389745         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 1|561            |
|01/01/2006 12:00:00 AM|01/2006|Terminal 2|98991          |
|01/01/2006 12:00:00 AM|01/2006|Terminal 2|163067         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 2|93672          |
|01/01/2006 12:00:00 AM|01/2006|Terminal 2|156751         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 3|121649         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 3|26585          |
|01/01/2006 12:00:00 AM|01/2006|Terminal 3|120111         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 3|60948          |
|01/01/2006 12:00:00 AM|01/2006|Terminal 4|381419         |
|01/01/2006 12:00:00 AM|01/2006|Terminal 4|68348          |
|01/01/2006 12:00:00 AM|01/2006|Terminal

In [27]:
from pyspark.sql.functions import sum, col

monthly_passengers = df3.groupBy("Month") \
    .agg(
        sum("Passenger_Count").alias("Total_Passengers")
    ) \
    .orderBy(
        col("Total_Passengers").desc()
    )

monthly_passengers.show()

+-------+----------------+
|  Month|Total_Passengers|
+-------+----------------+
|07/2017|         7910993|
|08/2017|         7674171|
|07/2016|         7621225|
|06/2017|         7470964|
|08/2016|         7344648|
|06/2016|         7230571|
|07/2015|         7063573|
|05/2017|         6933026|
|08/2015|         6902789|
|07/2014|         6709940|
|12/2016|         6640915|
|04/2017|         6640015|
|10/2016|         6629325|
|08/2014|         6595721|
|06/2015|         6579357|
|05/2016|         6571091|
|03/2017|         6525366|
|09/2016|         6375328|
|06/2014|         6373135|
|07/2013|         6292610|
+-------+----------------+
only showing top 20 rows


In [38]:
monthly_passengers.agg(count('Total_Passengers').alias('ans')).show()

+---+
|ans|
+---+
|140|
+---+

